# Temperature-programmed reduction: LabVIEW TCD rig

`TPRData` holds the detector signals (mV) and the temperature (°C) over time (s).
On this rig a reduction is a positive excursion of `tcd_h2`. The files are the
test fixtures under `tests/tpr/labview_tcd/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import lab_data_parser as ldp

In [ ]:
TPR = Path("..") / "tests" / "tpr" / "labview_tcd"
tpr_raw = ldp.TPRData.from_labview_tcd(TPR / "tpx3098_v2.txt")
print("signals:", tpr_raw.signals)
print("sample weight / mg:", tpr_raw.sample_weight_mg)
print("points:", tpr_raw.n_points)

In [ ]:
fig, (ax_t, ax_s) = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
ax_t.plot(tpr_raw.time / 60, tpr_raw.temperature)
ax_t.set_ylabel("temperature / °C")
ax_s.plot(tpr_raw.time / 60, tpr_raw.get_signal("tcd_h2"))
ax_s.set_ylabel("tcd signal / mV")
ax_s.set_xlabel("time / min")
plt.show()

## Baseline, per gram, integral

A linear baseline through two anchor times before and after the reduction,
then the signal is divided by the sample weight and integrated. The TCD is not
calibrated for H2, so the result is an area in mV s g^-1 that compares runs on
the same rig.

In [ ]:
end_s = float(tpr_raw.time[-1])
tpr = (
    tpr_raw
    .subtract_baseline("tcd_h2", "linear", anchor_points_s=[300, end_s - 300])
    .per_gram("tcd_h2")
)
print(f"consumption: {tpr.integrate('tcd_h2'):.4g} mV s g^-1")
print(tpr.format_history())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(tpr.time / 60, tpr.get_signal("tcd_h2"))
ax.set_xlabel("time / min")
ax.set_ylabel("tcd signal / mV g^-1")
plt.show()

## Heating rate

In [ ]:
tpr_raw.heating_rate()